# 01 · Run Pipeline & Experiments

This notebook executes experiment sweeps defined in `configs/*.yaml` and logs completed runs to:
- `results/runs/<user>.v2.csv` (summary metric and environment provenance for each run)
- `results/epochs/<user>.v2.csv` (epoch-by-epoch loss, accuracy, and timing curves)

In [ ]:
#Environment Setup and Paths
import os
import sys
from pathlib import Path

# Find repository root
REPO_DIR = Path.cwd()
while REPO_DIR != REPO_DIR.parent and not (REPO_DIR / "pyproject.toml").exists():
    REPO_DIR = REPO_DIR.parent

if not (REPO_DIR / "pyproject.toml").exists():
    raise RuntimeError(f"Could not find repository root with pyproject.toml from {Path.cwd()}")

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.chdir(REPO_DIR)

%load_ext autoreload
%autoreload 2

import torch
import satbench
from satbench.env import resolve_device, device_name

device = resolve_device("auto")
print(f"satbench version : {satbench.__version__}")
print(f"Repository root  : {REPO_DIR}")
print(f"Compute device   : {device_name(device)} ({device})")
print(f"PyTorch version  : {torch.__version__}")

## 1. Configure Experiment Settings

Set your username and select which configuration file to execute.
- `USER`: Used to namespace result files (e.g. `results/runs/<user>.v2.csv`). Each member appends only to their own CSV, preventing merge conflicts. Use your own name; `felix`, `manuel`, and `maxwell` already have results in the repo.
- `CONFIG_PATH`: Path to YAML experiment file.
- `OVERRIDES`: Optional dictionary to quickly test a smaller subset (e.g. specific models, fewer epochs, or a single seed) without editing the YAML file.

In [ ]:
# Set your username (used to namespace your result CSVs).
# Use your own name. maxwell, manuel, and felix already have results in the repo.
USER = os.environ.get("SATBENCH_USER", "").strip().lower()
if not USER:
    raise ValueError(
        "Set USER = \"yourname\" in this cell, or export SATBENCH_USER, before running."
    )

# Path to the experiment YAML config
# 1. Start with configs/smoke_test.yaml
# 2. ResNet sweep: configs/baseline_label_efficiency.yaml
# 3. ViT sweep:    configs/vit_label_efficiency.yaml
CONFIG_PATH = "configs/linear_probe_lr.yaml"

# Optional overrides (set to None to keep default values from the YAML file)
OVERRIDES = dict(
    epochs=None,           # e.g., 5 or 10
    seeds=None,            # e.g., [0] to test one seed, or [0, 1, 2]
    models=None,           # e.g., ["linear", "resnet50_random", "resnet50_imagenet"]
    train_fractions=None,  # e.g., [0.01, 0.05, 0.10]
    device="auto",         # "auto" | "cuda" | "mps" | "cpu"
)

# Filter out None overrides
active_overrides = {k: v for k, v in OVERRIDES.items() if v is not None}

In [ ]:
from satbench import ExperimentConfig

# Parse configuration and print summary grid
cfg = ExperimentConfig.from_yaml(
    CONFIG_PATH,
    user=USER,
    results_dir=str(REPO_DIR / "results"),
    data_dir=str(REPO_DIR / "data"),
    splits_dir=str(REPO_DIR / "splits"),
    **active_overrides
)
print(cfg.summary())

## 2. Run the Experiment Grid

The cell below runs every combination in the experiment grid (`model` × `train_fraction` × `seed`).
- If a run was already completed, it is automatically skipped.
- If you interrupt training, finished runs are preserved on disk. Re-running the cell will resume where it left off.

In [ ]:
from satbench import run_experiment

# Execute the experiment sweep
new_runs = run_experiment(cfg)

## 3. Review Run Results

Display the metrics from newly completed runs.

In [ ]:
if not new_runs.empty:
    display_cols = [
        "model", "train_fraction", "seed", "n_train", "best_epoch",
        "best_val_acc", "test_acc", "test_macro_f1", "total_time_s", "device_name"
    ]
    cols = [c for c in display_cols if c in new_runs.columns]
    display(new_runs[cols])
else:
    print("All runs for this configuration were already completed.")

## 4. Inspect Cumulative Results in `results/`

Check all completed runs recorded in `results/runs/` and `results/epochs/`.

In [ ]:
from satbench import load_runs, load_epochs

runs_df = load_runs(REPO_DIR / "results")
epochs_df = load_epochs(REPO_DIR / "results", runs_df)

print(f"Total completed runs recorded in results/ : {len(runs_df)}")
print(f"Total epoch rows recorded in results/     : {len(epochs_df)}")

if not runs_df.empty:
    summary_cols = ["run_id", "experiment", "user", "model", "train_fraction", "seed", "test_acc", "finished_at"]
    display(runs_df[summary_cols].tail(10))

## Next Steps

1. For full ResNet sweeps use `CONFIG_PATH = "configs/baseline_label_efficiency.yaml"`; for ViT use `configs/vit_label_efficiency.yaml`. Re-run Section 1 onward.
2. Open `notebooks/02_visualize_results.ipynb` to analyze label efficiency curves, learning trajectories, and confusion matrices.